# Demo 3: Kafka Streaming Pipeline

BME Adatmérnökség – 2. hét

Ebben a demóban:
1. Kafka topic létrehozás és konfiguráció
2. Python producer: szimulált e-commerce események
3. Python consumer: valós idejű feldolgozás
4. Ablakos aggregáció (windowed aggregation)
5. Schema Registry és Avro

In [1]:
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import Producer, Consumer, KafkaError
import json
import time
import random
from datetime import datetime
import requests

# Kafka Admin
admin = AdminClient({'bootstrap.servers': 'kafka:9092'})

# Topic-ok létrehozása
topics_to_create = [
    NewTopic("demo3-orders", num_partitions=3, replication_factor=1),
    NewTopic("demo3-aggregated", num_partitions=1, replication_factor=1),
]

print("=== Kafka Topic-ok létrehozása ===")
print()
fs = admin.create_topics(topics_to_create)
for topic, f in fs.items():
    try:
        f.result()
        print(f"  Létrehozva: {topic}")
    except Exception as e:
        if "TOPIC_ALREADY_EXISTS" in str(e):
            print(f"  Már létezik: {topic}")
        else:
            print(f"  Hiba ({topic}): {e}")

# Meglévő topic-ok
print("\n=== Összes topic ===")
metadata = admin.list_topics(timeout=10)
for t in sorted(metadata.topics.keys()):
    if not t.startswith('_'):
        partitions = len(metadata.topics[t].partitions)
        print(f"  {t} ({partitions} partíció)")

=== Kafka Topic-ok létrehozása ===

  Létrehozva: demo3-orders
  Létrehozva: demo3-aggregated

=== Összes topic ===
  debezium_configs (1 partíció)
  debezium_offsets (25 partíció)
  debezium_statuses (5 partíció)
  demo3-aggregated (1 partíció)
  demo3-orders (3 partíció)


## 1. Kafka Producer: E-commerce események

Szimulált webshop eseményeket küldünk Kafkába: rendelés létrehozás, fizetés, szállítás.

In [2]:
# Kafka Producer
producer = Producer({
    'bootstrap.servers': 'kafka:9092',
    'client.id': 'demo3-producer',
})

# Szimulált események
customers = [f"customer_{i}" for i in range(1, 11)]
products = ["Laptop", "Telefon", "Fejhallgató", "Monitor", "Billentyűzet", "Egér", "SSD", "Tablet"]
event_types = ["order_created", "payment_received", "item_shipped", "item_delivered"]

def delivery_report(err, msg):
    if err:
        print(f"  HIBA: {err}")

print("=== Események küldése ===")
print()
sent_count = 0

for i in range(30):
    event = {
        "event_id": f"evt-{i+1:04d}",
        "event_type": random.choice(event_types),
        "customer_id": random.choice(customers),
        "product": random.choice(products),
        "amount": random.randint(5000, 500000),
        "timestamp": datetime.now().isoformat(),
    }
    
    # Partition key = customer_id (azonos ügyfél azonos partícióba)
    producer.produce(
        topic="demo3-orders",
        key=event["customer_id"].encode('utf-8'),
        value=json.dumps(event).encode('utf-8'),
        callback=delivery_report,
    )
    sent_count += 1
    
    if (i + 1) % 10 == 0:
        producer.flush()
        print(f"  Küldve: {sent_count} esemény")

producer.flush()
print(f"\nÖsszesen {sent_count} esemény küldve a 'demo3-orders' topic-ba.")

=== Események küldése ===

  Küldve: 10 esemény
  Küldve: 20 esemény
  Küldve: 30 esemény

Összesen 30 esemény küldve a 'demo3-orders' topic-ba.


## 2. Kafka Consumer: Események olvasása

A consumer group automatikusan elosztja a partíciókat a csoport tagjai között.

In [3]:
# Kafka Consumer
consumer = Consumer({
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'demo3-consumer-group',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
})

consumer.subscribe(['demo3-orders'])
print("=== Események olvasása ===")
print()

messages = []
empty_polls = 0

while len(messages) < 30 and empty_polls < 10:
    msg = consumer.poll(2.0)
    
    if msg is None:
        empty_polls += 1
        continue
    
    if msg.error():
        if msg.error().code() == KafkaError._PARTITION_EOF:
            continue
        print(f"Hiba: {msg.error()}")
        break
    
    empty_polls = 0
    event = json.loads(msg.value().decode('utf-8'))
    messages.append(event)
    
    print(f"  [{len(messages):>2d}] Partíció: {msg.partition()}, Offset: {msg.offset()} | "
          f"{event['event_type']:20s} | {event['customer_id']} | {event['amount']:>7,} Ft")

consumer.close()
print(f"\nÖsszesen {len(messages)} üzenet feldolgozva.")

=== Események olvasása ===

  [ 1] Partíció: 1, Offset: 0 | order_created        | customer_10 |  62,434 Ft
  [ 2] Partíció: 1, Offset: 1 | order_created        | customer_10 | 265,526 Ft
  [ 3] Partíció: 1, Offset: 2 | payment_received     | customer_7 | 188,469 Ft
  [ 4] Partíció: 1, Offset: 3 | order_created        | customer_10 | 307,617 Ft
  [ 5] Partíció: 1, Offset: 4 | item_delivered       | customer_7 | 149,010 Ft
  [ 6] Partíció: 1, Offset: 5 | payment_received     | customer_7 | 201,246 Ft
  [ 7] Partíció: 1, Offset: 6 | item_delivered       | customer_10 | 294,982 Ft
  [ 8] Partíció: 1, Offset: 7 | order_created        | customer_2 | 366,972 Ft
  [ 9] Partíció: 1, Offset: 8 | payment_received     | customer_10 |  54,214 Ft
  [10] Partíció: 1, Offset: 9 | order_created        | customer_10 | 289,639 Ft
  [11] Partíció: 0, Offset: 0 | item_delivered       | customer_9 | 394,159 Ft
  [12] Partíció: 0, Offset: 1 | item_delivered       | customer_5 |  30,939 Ft
  [13] Partíció: 0

## 3. Ablakos aggregáció (Windowed Aggregation)

Egyszerű tumbling window: percenkénti összesítés az eseményekből.

In [4]:
# Tumbling Window aggregáció
from collections import defaultdict

print("=== Tumbling Window: eseménytípus szerinti összesítés ===")
print()

# Ablak mérete: eseménytípusonkénti összesítés
type_stats = defaultdict(lambda: {"count": 0, "total_amount": 0, "customers": set()})

for event in messages:
    et = event['event_type']
    type_stats[et]["count"] += 1
    type_stats[et]["total_amount"] += event['amount']
    type_stats[et]["customers"].add(event['customer_id'])

print(f"  {'Eseménytípus':25s} {'Darab':>6s} {'Összeg':>12s} {'Ügyfelek':>8s}")
print(f"  {'-'*55}")
for et, stats in sorted(type_stats.items()):
    print(f"  {et:25s} {stats['count']:>6d} {stats['total_amount']:>12,} Ft {len(stats['customers']):>8d}")

# Ügyfélenkénti összesítés
print(f"\n=== Sliding Window: top ügyfelek ===")
print()
customer_stats = defaultdict(lambda: {"count": 0, "total": 0})
for event in messages:
    cid = event['customer_id']
    customer_stats[cid]["count"] += 1
    customer_stats[cid]["total"] += event['amount']

top = sorted(customer_stats.items(), key=lambda x: x[1]['total'], reverse=True)[:5]
print(f"  {'Ügyfél':15s} {'Események':>10s} {'Összeg':>12s}")
print(f"  {'-'*40}")
for cid, stats in top:
    print(f"  {cid:15s} {stats['count']:>10d} {stats['total']:>12,} Ft")

=== Tumbling Window: eseménytípus szerinti összesítés ===

  Eseménytípus               Darab       Összeg Ügyfelek
  -------------------------------------------------------
  item_delivered                 9    2,337,413 Ft        6
  item_shipped                   4      889,415 Ft        4
  order_created                  9    2,575,805 Ft        6
  payment_received               8    2,081,033 Ft        5

=== Sliding Window: top ügyfelek ===

  Ügyfél           Események       Összeg
  ----------------------------------------
  customer_9               5    1,865,879 Ft
  customer_10              6    1,274,412 Ft
  customer_4               3    1,001,899 Ft
  customer_8               3      906,241 Ft
  customer_5               5      858,562 Ft


## 4. Producer-Consumer összekötés: valós idejű feldolgozás

Küldés és feldolgozás párhuzamosan, aggregált eredmény visszaírása Kafkába.

In [5]:
# Valós idejű pipeline: produce → consume → aggregate → produce
import threading

aggregated_results = []
stop_flag = threading.Event()

def consumer_worker():
    """Consumer: olvas és aggregál"""
    c = Consumer({
        'bootstrap.servers': 'kafka:9092',
        'group.id': 'demo3-realtime',
        'auto.offset.reset': 'latest',
        'enable.auto.commit': True,
    })
    c.subscribe(['demo3-orders'])
    
    window = []
    window_size = 5  # 5 üzenetenként aggregálunk
    
    while not stop_flag.is_set():
        msg = c.poll(1.0)
        if msg and not msg.error():
            event = json.loads(msg.value().decode('utf-8'))
            window.append(event)
            
            if len(window) >= window_size:
                agg = {
                    "window_end": datetime.now().isoformat(),
                    "event_count": len(window),
                    "total_amount": sum(e['amount'] for e in window),
                    "avg_amount": sum(e['amount'] for e in window) // len(window),
                    "types": dict(defaultdict(int, {e['event_type']: 1 for e in window})),
                }
                aggregated_results.append(agg)
                window = []
    c.close()

# Consumer indítása háttérszálban
thread = threading.Thread(target=consumer_worker)
thread.start()
time.sleep(2)

# Producer: újabb események
p = Producer({'bootstrap.servers': 'kafka:9092'})
print("=== Valós idejű pipeline ===")
print()
print("Események küldése és feldolgozása párhuzamosan...")
print()

for i in range(20):
    event = {
        "event_id": f"rt-{i+1:04d}",
        "event_type": random.choice(event_types),
        "customer_id": random.choice(customers),
        "product": random.choice(products),
        "amount": random.randint(5000, 500000),
        "timestamp": datetime.now().isoformat(),
    }
    p.produce("demo3-orders", key=event["customer_id"].encode(), value=json.dumps(event).encode())
    if (i + 1) % 5 == 0:
        p.flush()
        time.sleep(1)
        print(f"  Küldve: {i+1} esemény, Aggregált ablakok: {len(aggregated_results)}")

p.flush()
time.sleep(3)
stop_flag.set()
thread.join(timeout=5)

print(f"\n=== Aggregált ablakok ({len(aggregated_results)} db) ===")
print()
for i, agg in enumerate(aggregated_results):
    print(f"  Ablak #{i+1}: {agg['event_count']} esemény, összeg: {agg['total_amount']:,} Ft, átlag: {agg['avg_amount']:,} Ft")

=== Valós idejű pipeline ===

Események küldése és feldolgozása párhuzamosan...

  Küldve: 5 esemény, Aggregált ablakok: 0
  Küldve: 10 esemény, Aggregált ablakok: 0
  Küldve: 15 esemény, Aggregált ablakok: 1
  Küldve: 20 esemény, Aggregált ablakok: 2

=== Aggregált ablakok (2 db) ===

  Ablak #1: 5 esemény, összeg: 604,382 Ft, átlag: 120,876 Ft
  Ablak #2: 5 esemény, összeg: 1,646,131 Ft, átlag: 329,226 Ft


## 5. Schema Registry és Avro

Séma regisztrálása és sémaevolúció demonstrálása.

In [6]:
# Schema Registry
SR_URL = "http://schema-registry:8081"

# Avro séma definiálása
order_schema_v1 = {
    "type": "record",
    "name": "OrderEvent",
    "namespace": "com.bme.dataeng",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "event_type", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "product", "type": "string"},
        {"name": "amount", "type": "int"},
        {"name": "timestamp", "type": "string"}
    ]
}

# Séma regisztrálása
print("=== Schema Registry: Avro séma regisztrálás ===")
print()
resp = requests.post(
    f"{SR_URL}/subjects/demo3-orders-value/versions",
    headers={"Content-Type": "application/vnd.schemaregistry.v1+json"},
    json={"schemaType": "AVRO", "schema": json.dumps(order_schema_v1)}
)
print(f"V1 regisztrálva: schema ID = {resp.json().get('id', resp.json())}")

# Séma lekérdezés
resp = requests.get(f"{SR_URL}/subjects/demo3-orders-value/versions/latest")
schema_info = resp.json()
print(f"\nAktuális séma:")
print(f"  Subject: {schema_info.get('subject')}")
print(f"  Version: {schema_info.get('version')}")
print(f"  Schema ID: {schema_info.get('id')}")

=== Schema Registry: Avro séma regisztrálás ===

V1 regisztrálva: schema ID = 1

Aktuális séma:
  Subject: demo3-orders-value
  Version: 1
  Schema ID: 1


In [7]:
# Sémaevolúció: új mező hozzáadása (backward compatible)
order_schema_v2 = {
    "type": "record",
    "name": "OrderEvent",
    "namespace": "com.bme.dataeng",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "event_type", "type": "string"},
        {"name": "customer_id", "type": "string"},
        {"name": "product", "type": "string"},
        {"name": "amount", "type": "int"},
        {"name": "timestamp", "type": "string"},
        {"name": "currency", "type": ["null", "string"], "default": None}  # ÚJ mező, opcionális
    ]
}

print("=== Sémaevolúció: V2 (új 'currency' mező) ===")
print()

# Kompatibilitás ellenőrzése
resp = requests.post(
    f"{SR_URL}/compatibility/subjects/demo3-orders-value/versions/latest",
    headers={"Content-Type": "application/vnd.schemaregistry.v1+json"},
    json={"schemaType": "AVRO", "schema": json.dumps(order_schema_v2)}
)
compat = resp.json()
print(f"Kompatibilitás ellenőrzés: {'KOMPATIBILIS' if compat.get('is_compatible', False) else 'NEM KOMPATIBILIS'}")

# V2 regisztrálása
resp = requests.post(
    f"{SR_URL}/subjects/demo3-orders-value/versions",
    headers={"Content-Type": "application/vnd.schemaregistry.v1+json"},
    json={"schemaType": "AVRO", "schema": json.dumps(order_schema_v2)}
)
print(f"V2 regisztrálva: schema ID = {resp.json().get('id', resp.json())}")

# Verziók listázása
resp = requests.get(f"{SR_URL}/subjects/demo3-orders-value/versions")
print(f"\nÖsszes verzió: {resp.json()}")

# Breaking change teszt
print("\n=== Breaking change teszt ===")
order_schema_breaking = {
    "type": "record",
    "name": "OrderEvent",
    "namespace": "com.bme.dataeng",
    "fields": [
        {"name": "event_id", "type": "string"},
        {"name": "amount", "type": "string"},  # int → string: BREAKING!
    ]
}
resp = requests.post(
    f"{SR_URL}/compatibility/subjects/demo3-orders-value/versions/latest",
    headers={"Content-Type": "application/vnd.schemaregistry.v1+json"},
    json={"schemaType": "AVRO", "schema": json.dumps(order_schema_breaking)}
)
compat = resp.json()
print(f"Breaking change: {'KOMPATIBILIS' if compat.get('is_compatible', False) else 'ELUTASÍTVA (nem kompatibilis)'}")

=== Sémaevolúció: V2 (új 'currency' mező) ===

Kompatibilitás ellenőrzés: KOMPATIBILIS
V2 regisztrálva: schema ID = 2

Összes verzió: [1, 2]

=== Breaking change teszt ===
Breaking change: ELUTASÍTVA (nem kompatibilis)


## 6. Kézbesítési garanciák demonstrálása

At-least-once vs at-most-once producer beállítások.

In [8]:
# Kézbesítési garanciák
print("=== Kézbesítési garanciák ===")
print()

configs = {
    "At-most-once (legfeljebb egyszer)": {
        'bootstrap.servers': 'kafka:9092',
        'acks': '0',  # Nem vár nyugtázásra
    },
    "At-least-once (legalább egyszer)": {
        'bootstrap.servers': 'kafka:9092',
        'acks': 'all',  # Minden replika nyugtázza
        'retries': 3,
    },
    "Exactly-once (pontosan egyszer)": {
        'bootstrap.servers': 'kafka:9092',
        'acks': 'all',
        'enable.idempotence': True,  # Idempotens producer
    },
}

for name, config in configs.items():
    p = Producer(config)
    start = time.time()
    for i in range(10):
        p.produce("demo3-orders", value=json.dumps({"test": i, "guarantee": name}).encode())
    p.flush()
    elapsed = (time.time() - start) * 1000
    print(f"  {name}:")
    print(f"    Config: acks={config.get('acks')}, idempotent={config.get('enable.idempotence', False)}")
    print(f"    10 üzenet küldése: {elapsed:.1f} ms")
    print()

=== Kézbesítési garanciák ===

  At-most-once (legfeljebb egyszer):
    Config: acks=0, idempotent=False
    10 üzenet küldése: 6.0 ms

  At-least-once (legalább egyszer):
    Config: acks=all, idempotent=False
    10 üzenet küldése: 15.0 ms



%4|1771843192.463|GETPID|rdkafka#producer-8| [thrd:main]: Failed to acquire idempotence PID from broker kafka:9092/1: Broker: Coordinator load in progress: retrying


  Exactly-once (pontosan egyszer):
    Config: acks=all, idempotent=True
    10 üzenet küldése: 1014.2 ms



In [9]:
# Topic-ok törlése (opcionális)
# admin.delete_topics(["demo3-orders", "demo3-aggregated"])

print("=== Demo 3 összefoglalás ===")
print("1. Kafka topic-ok: particionálás, consumer group-ok")
print("2. Producer: események küldése partition key-jel")
print("3. Consumer: üzenetek olvasása és feldolgozása")
print("4. Ablakos aggregáció: tumbling window összesítés")
print("5. Schema Registry: Avro séma regisztrálás és evolúció")
print("6. Kézbesítési garanciák: at-most/at-least/exactly-once")
print("\nA kafka-ui felületen (http://localhost:8080) vizuálisan is áttekinthetők")
print("a topic-ok, üzenetek és consumer group-ok.")
print("\n=== Week 02 demók vége ===")

=== Demo 3 összefoglalás ===
1. Kafka topic-ok: particionálás, consumer group-ok
2. Producer: események küldése partition key-jel
3. Consumer: üzenetek olvasása és feldolgozása
4. Ablakos aggregáció: tumbling window összesítés
5. Schema Registry: Avro séma regisztrálás és evolúció
6. Kézbesítési garanciák: at-most/at-least/exactly-once

A kafka-ui felületen (http://localhost:8080) vizuálisan is áttekinthetők
a topic-ok, üzenetek és consumer group-ok.

=== Week 02 demók vége ===
